In [1]:
%pip install pypdf langchain-text-splitters chromadb sentence-transformers ollama

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import warnings
from pathlib import Path
from pypdf import PdfReader

# Silence progress bars / warnings that can hang the notebook UI
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

# Resolve the project root whether this notebook is run from the repo root
# or from notebooks/ directly, so paths work on any machine.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").is_dir() and (PROJECT_ROOT.parent / "data").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
CHROMA_PATH = PROJECT_ROOT / "chroma_db"


def extract_text_from_pdfs(folder_path: Path) -> list[dict]:
    extracted_data = []

    pdf_files = sorted(
        f for f in folder_path.iterdir()
        if f.is_file() and f.suffix.lower() == ".pdf"
    )

    if not pdf_files:
        print(f"No PDF files found in: {folder_path}")
        return extracted_data

    print(f"Found {len(pdf_files)} PDF file(s). Extracting text...\n")

    for pdf_path in pdf_files:
        try:
            reader = PdfReader(pdf_path)
            total_pages = len(reader.pages)

            for page_num, page in enumerate(reader.pages, start=1):
                page_text = page.extract_text() or ""
                cleaned_text = page_text.strip()

                if cleaned_text:
                    extracted_data.append({
                        "file_name": pdf_path.name,
                        "file_path": str(pdf_path),
                        "page_number": page_num,
                        "total_pages": total_pages,
                        "text": cleaned_text
                    })
            print(f"✓ Processed: {pdf_path.name} ({total_pages} pages)")
        except Exception as e:
            print(f"✗ Failed to read {pdf_path.name}: {e}")

    return extracted_data

records = extract_text_from_pdfs(DATA_DIR)
print(f"\nTotal raw pages extracted: {len(records)}")

Found 7 PDF file(s). Extracting text...

✓ Processed: 01_Resume_Writing_Best_Practices.pdf (3 pages)
✓ Processed: 02_How_to_Analyze_a_Job_Description.pdf (3 pages)
✓ Processed: 03_Behavioral_Interview_Framework.pdf (3 pages)
✓ Processed: 04_Technical_Interview_and_System_Design.pdf (3 pages)
✓ Processed: 05_Salary_Negotiation_Playbook.pdf (2 pages)
✓ Processed: 06_Career_Growth_and_Personal_Branding.pdf (2 pages)
✓ Processed: 07_Data_Roles_Roadmaps_and_Tools.pdf (3 pages)

Total raw pages extracted: 19


In [3]:
records[0]

{'file_name': '01_Resume_Writing_Best_Practices.pdf',
 'file_path': 'C:\\Users\\Ahmed\\Desktop\\NRAG\\data\\01_Resume_Writing_Best_Practices.pdf',
 'page_number': 1,
 'total_pages': 3,
 'text': 'Resume Writing Best Practices\n CareerForge Knowledge Base · Volume 1 · 2025-2026 Edition\nTarget reader: Anyone actively applying to roles who wants a resume that survives both AI/ATS screening\nand a human recruiter\'s 7-second skim.\n1. The Two Readers Every Resume Has\nEvery resume submitted online today is read twice: once by software (an Applicant Tracking System, often\nlayered with an AI scoring or summarization step), and once -- if it passes -- by a human recruiter who\ntypically spends under 10 seconds on a first pass. Optimizing for one at the expense of the other backfires.\nA resume stuffed with keywords may pass a naive parser but reads as robotic and gets rejected by a\nperson; a beautifully designed two-column resume may impress a human but fail to parse at all.\nWhat actually 

In [4]:
import re

def clean_document_text(text: str) -> str:
    # 1. Strip repeated disclaimer/legal boilerplate that trails each page
    disclaimer_pattern = r"This guide reflects widely-observed hiring practices.*?industry, and location\."
    text = re.sub(disclaimer_pattern, "", text, flags=re.DOTALL | re.IGNORECASE)

    # 2. Remove stray control characters left over from PDF extraction (\x7f)
    text = text.replace('\x7f', '')

    # 3. Rejoin words split across a line break by a hyphen (e.g. "re-\nsponsible" -> "responsible")
    text = re.sub(r'(\w+)-\n(\w+)', r'\1\2', text)

    # 4. Normalize all bullet glyphs (■, •, ▪, ►, *) to a single "- " marker
    text = re.sub(r'(?:[\r\n]+\s*)+[■•▪►\*\-]\s*', r'\n- ', text)

    # 5. Clean up orphaned bullet markers left by the normalization above
    text = re.sub(r'\n[-*]\s*\n', '\n- ', text)

    # 6. Collapse extra whitespace and blank lines
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)

    return text.strip()

def preprocess_records(raw_records: list[dict]) -> list[dict]:
    cleaned = []
    for record in raw_records:
        cleaned_content = clean_document_text(record["text"])
        if len(cleaned_content) > 30:
            cleaned.append({
                "file_name": record["file_name"],
                "file_path": record["file_path"],
                "page_number": record["page_number"],
                "total_pages": record["total_pages"],
                "text": cleaned_content
            })
    print(f"✓ Cleaned {len(cleaned)} pages successfully.")
    return cleaned

cleaned_records = preprocess_records(records)

✓ Cleaned 18 pages successfully.


In [5]:
cleaned_records[:3]

[{'file_name': '01_Resume_Writing_Best_Practices.pdf',
  'file_path': 'C:\\Users\\Ahmed\\Desktop\\NRAG\\data\\01_Resume_Writing_Best_Practices.pdf',
  'page_number': 1,
  'total_pages': 3,
  'text': 'Resume Writing Best Practices\n CareerForge Knowledge Base · Volume 1 · 2025-2026 Edition\nTarget reader: Anyone actively applying to roles who wants a resume that survives both AI/ATS screening\nand a human recruiter\'s 7-second skim.\n1. The Two Readers Every Resume Has\nEvery resume submitted online today is read twice: once by software (an Applicant Tracking System, often\nlayered with an AI scoring or summarization step), and once -- if it passes -- by a human recruiter who\ntypically spends under 10 seconds on a first pass. Optimizing for one at the expense of the other backfires.\nA resume stuffed with keywords may pass a naive parser but reads as robotic and gets rejected by a\nperson; a beautifully designed two-column resume may impress a human but fail to parse at all.\nWhat actu

In [6]:
import hashlib

from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,
    chunk_overlap=180,
    separators=["\n\n", "\n", ". ", " "]
)

def make_chunk_id(file_name: str, page_number: int, text: str) -> str:
    # Content hash (not a running counter) -> re-running this notebook on
    # unchanged PDFs always produces the same IDs, so the indexing step
    # below can skip chunks that are already in the database instead of
    # re-embedding everything from scratch every time.
    digest = hashlib.sha256(f"{file_name}|{page_number}|{text}".encode("utf-8")).hexdigest()[:16]
    return f"{file_name}_p{page_number}_{digest}"

chunks = []

for record in cleaned_records:
    splits = text_splitter.split_text(record["text"])
    for s in splits:
        stripped = s.strip()
        if len(stripped) > 40:
            chunks.append({
                "chunk_id": make_chunk_id(record["file_name"], record["page_number"], stripped),
                "file_name": str(record["file_name"]),
                "page_number": int(record["page_number"]),
                "text": stripped
            })

print(f"✓ Generated {len(chunks)} cohesive chunks with context overlap.")

✓ Generated 78 cohesive chunks with context overlap.


In [7]:
chunks[:3]

[{'chunk_id': '01_Resume_Writing_Best_Practices.pdf_p1_5c97f00854eb0646',
  'file_name': '01_Resume_Writing_Best_Practices.pdf',
  'page_number': 1,
  'text': "Resume Writing Best Practices\n CareerForge Knowledge Base · Volume 1 · 2025-2026 Edition\nTarget reader: Anyone actively applying to roles who wants a resume that survives both AI/ATS screening\nand a human recruiter's 7-second skim.\n1. The Two Readers Every Resume Has\nEvery resume submitted online today is read twice: once by software (an Applicant Tracking System, often\nlayered with an AI scoring or summarization step), and once -- if it passes -- by a human recruiter who\ntypically spends under 10 seconds on a first pass. Optimizing for one at the expense of the other backfires.\nA resume stuffed with keywords may pass a naive parser but reads as robotic and gets rejected by a"},
 {'chunk_id': '01_Resume_Writing_Best_Practices.pdf_p1_e2c30f9d325cdf4f',
  'file_name': '01_Resume_Writing_Best_Practices.pdf',
  'page_number'

In [8]:
import chromadb
from chromadb.utils import embedding_functions

os.makedirs(CHROMA_PATH, exist_ok=True)

client = chromadb.PersistentClient(path=str(CHROMA_PATH))

# Lightweight local embedding model -- no API key, runs on CPU
embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

collection = client.get_or_create_collection(
    name="career_knowledge_base",
    embedding_function=embedding_fn,
    metadata={"hnsw:space": "cosine"}
)

# Incremental indexing: only embed chunks whose ID isn't already stored.
# Re-running this cell after adding a new PDF costs almost nothing instead
# of re-embedding the whole knowledge base every time.
batch_size = 64
all_ids = [c["chunk_id"] for c in chunks]
existing_ids = set()
for i in range(0, len(all_ids), batch_size):
    batch_ids = all_ids[i:i + batch_size]
    existing_ids.update(collection.get(ids=batch_ids, include=[])["ids"])

new_chunks = [c for c in chunks if c["chunk_id"] not in existing_ids]

for i in range(0, len(new_chunks), batch_size):
    batch = new_chunks[i:i + batch_size]
    collection.add(
        ids=[c["chunk_id"] for c in batch],
        documents=[c["text"] for c in batch],
        metadatas=[{"file_name": c["file_name"], "page_number": c["page_number"]} for c in batch]
    )

print(f"✓ Indexed {len(new_chunks)} new chunk(s). Total chunks in DB: {collection.count()}")

✓ Indexed 0 new chunk(s). Total chunks in DB: 78


In [9]:
def retrieve_relevant_chunks(query: str, n_results: int = 4) -> list[dict]:
    """Retrieve the top semantically-relevant chunks for a query, with metadata."""
    results = collection.query(
        query_texts=[query],
        n_results=n_results,
        include=["documents", "metadatas", "distances"]
    )

    retrieved_items = []
    docs = results["documents"][0]
    metas = results["metadatas"][0]
    distances = results["distances"][0]

    for doc, meta, dist in zip(docs, metas, distances):
        similarity = round((1 - dist) * 100, 2)
        retrieved_items.append({
            "text": doc,
            "file_name": meta["file_name"],
            "page_number": meta["page_number"],
            "similarity": similarity
        })

    return retrieved_items

def search_and_display(query: str, n_results: int = 4):
    """Pretty-print search results for quick manual inspection."""
    print(f"\nSearching for: '{query}'")
    print("=" * 70)

    hits = retrieve_relevant_chunks(query, n_results=n_results)
    for idx, hit in enumerate(hits, start=1):
        print(f"Result #{idx} | Similarity: {hit['similarity']}% | Source: {hit['file_name']} (Page {hit['page_number']})")
        print("-" * 70)
        print(hit["text"])
        print("=" * 70)

test_query = "How should I structure my resume bullet points to show measurable impact?"
search_and_display(test_query, n_results=4)


Searching for: 'How should I structure my resume bullet points to show measurable impact?'
Result #1 | Similarity: 58.56% | Source: 01_Resume_Writing_Best_Practices.pdf (Page 3)
----------------------------------------------------------------------
Mirror the job description's language truthfully in your summary and skills section (see the companion
"How to Analyze a Job Description" guide for the extraction process).

Reorder your bullets so the most relevant achievements for *this* role appear first within each job
entry.

Run your tailored resume through a free ATS/keyword checker against the specific posting before
submitting, if one is available to you.
Result #2 | Similarity: 53.86% | Source: 01_Resume_Writing_Best_Practices.pdf (Page 2)
----------------------------------------------------------------------
Projects / Portfolio (optional but increasingly expected in tech) -- a short section linking to GitHub
repos, a portfolio site, or published work.
3. Writing Bullets That Act

In [10]:
import re

import ollama

OLLAMA_MODEL = "llama3.2:3b"

def build_context(retrieved_chunks: list[dict]) -> str:
    """Format retrieved chunks with source metadata for the prompt.

    Deliberately does NOT label chunks as "Document [1]", "Document [2]",
    etc. -- earlier versions of this prompt did, and the model would then
    cite "(Document [2], Page 2)" in its answer instead of the real file
    name, defeating the whole point of the citation rule below.
    """
    context_blocks = []
    for chunk in retrieved_chunks:
        block = (
            f"--- Source: {chunk['file_name']} (Page {chunk['page_number']}) ---\n"
            f"{chunk['text'].strip()}"
        )
        context_blocks.append(block)

    return "\n\n".join(context_blocks)


SYSTEM_INSTRUCTION = (
    "You are an expert career consultant. Answer the user's question directly and "
    "comprehensively using ONLY the provided context.\n"
    "Strict Rules:\n"
    "1. NEVER invent, extrapolate, or fabricate any examples. If an example is provided "
    "in the text, quote or adapt ONLY that exact example.\n"
    "2. Present any formula or framework given in the text (e.g. the X-Y-Z bullet formula) "
    "completely, along with its accompanying rules.\n"
    "3. Every paragraph or piece of advice MUST end with an explicit source citation in "
    "the format: (exact_file_name.pdf, Page X), using the real file name shown in the "
    "'Source:' header above each context block -- never a placeholder like 'Document [1]'.\n"
    "4. If the context does not contain enough information to answer, say so explicitly "
    "instead of guessing."
)

_PAREN_PATTERN = re.compile(r"\(([^()]+)\)")
_FILE_PATTERN = re.compile(r"([\w\-]+\.pdf)", re.IGNORECASE)
_PAGE_PATTERN = re.compile(r"Page\s*(\d+)", re.IGNORECASE)


def find_unverifiable_citations(answer: str, retrieved_chunks: list[dict]) -> list[str]:
    """Flag citations in the answer that don't match any retrieved (file, page).

    Looks for a *.pdf filename and a "Page N" inside each parenthetical
    independently (rather than requiring one exact format), so a harmless
    variation like "(Source: file.pdf, Page 3)" isn't a false positive --
    only a wrong/hallucinated file name or page number is flagged. This is
    what would have caught the earlier "(Document [2], Page 2)" bug
    automatically instead of relying on a human noticing it.
    """
    valid_pairs = {(c["file_name"].lower(), c["page_number"]) for c in retrieved_chunks}
    problems = []
    for paren_match in _PAREN_PATTERN.finditer(answer):
        content = paren_match.group(1)
        file_match = _FILE_PATTERN.search(content)
        page_match = _PAGE_PATTERN.search(content)
        if not (file_match and page_match):
            continue  # not a citation-shaped parenthetical, e.g. "(Y)" in the X-Y-Z formula
        file_name, page = file_match.group(1).lower(), int(page_match.group(1))
        if (file_name, page) not in valid_pairs:
            problems.append(paren_match.group(0))
    if re.search(r"\bDocument\s*\[\d+\]", answer, re.IGNORECASE):
        problems.append("placeholder citation (e.g. 'Document [1]') instead of a real file name")
    return problems


def generate_rag_response(query: str, n_results: int = 5) -> dict:
    retrieved_chunks = retrieve_relevant_chunks(query, n_results=n_results)

    if not retrieved_chunks:
        return {
            "answer": "No relevant information was found in the knowledge base for this question.",
            "sources": [],
            "citation_warnings": [],
        }

    formatted_context = build_context(retrieved_chunks)
    user_message = f"""Context Documents:
{formatted_context}

Question: {query}

Provide a structured, helpful answer based strictly on the context above:"""

    print(f"Generating answer using local {OLLAMA_MODEL}...")

    try:
        response = ollama.chat(
            model=OLLAMA_MODEL,
            messages=[
                {"role": "system", "content": SYSTEM_INSTRUCTION},
                {"role": "user", "content": user_message}
            ],
            options={"temperature": 0.1}
        )
    except Exception as exc:
        return {
            "answer": (
                f"Could not reach Ollama model '{OLLAMA_MODEL}'. "
                f"Make sure `ollama serve` is running and the model is pulled "
                f"(`ollama pull {OLLAMA_MODEL}`). Details: {exc}"
            ),
            "sources": retrieved_chunks,
            "citation_warnings": [],
        }

    answer = response["message"]["content"]
    citation_warnings = find_unverifiable_citations(answer, retrieved_chunks)

    return {"answer": answer, "sources": retrieved_chunks, "citation_warnings": citation_warnings}

In [11]:
# ====================================================
# End-to-end test
# ====================================================
test_query = "How should I structure my resume bullet points to show measurable impact?"

result = generate_rag_response(test_query, n_results=4)

print("\n" + "=" * 70)
print("🤖 Final Local RAG Response:")
print("=" * 70)
print(result["answer"])

if result["citation_warnings"]:
    print("\n⚠️  Unverifiable citation(s) detected:")
    for warning in result["citation_warnings"]:
        print(f"   - {warning}")

Generating answer using local llama3.2:3b...

🤖 Final Local RAG Response:
To structure your resume bullet points to show measurable impact, follow the X-Y-Z formula:

Accomplished [X], measured by [Y], by doing [Z].

This formula requires you to:

1. Identify the accomplishment (X)
2. Specify the metric or outcome (Y) that demonstrates the impact
3. Describe the action or effort taken to achieve the outcome (Z)

For example:

* "Reduced customer churn by 18% (Y) by redesigning the onboarding email sequence (Z), resulting in $240K in retained annual revenue (X)."

This structure helps to clearly convey the impact of your work and make your achievements more tangible to the reader.


In [12]:
# ====================================================
# Ask your own question
# ====================================================
my_question = "What's a good salary negotiation tactic?"  # edit this line and re-run

result = generate_rag_response(my_question, n_results=4)

print("\n" + "=" * 70)
print(f"🤖 Answer to: '{my_question}'")
print("=" * 70)
print(result["answer"])

print("\n" + "-" * 70)
print("Sources used:")
for src in result["sources"]:
    print(f"  - {src['file_name']} (Page {src['page_number']}, similarity {src['similarity']}%)")

if result["citation_warnings"]:
    print("\n⚠️  Unverifiable citation(s) detected:")
    for warning in result["citation_warnings"]:
        print(f"   - {warning}")

Generating answer using local llama3.2:3b...

🤖 Answer to: 'What's a good salary negotiation tactic?'
According to the 05_Salary_Negotiation_Playbook.pdf (Page 1), a good salary negotiation tactic is to "Know Your Number Before Any Conversation" (Source: 05_Salary_Negotiation_Playbook.pdf, Page 1). This involves:

1. Researching market rate from multiple sources (salary aggregation sites, industry surveys, professional network) and triangulating rather than trusting a single number.
2. Factoring in specific variables such as location, company size/stage, years of experience, and specialized/high-demand skills.
3. Deciding on a target salary, walk-away minimum, and ideal number above target before any conversation starts.
4. Understanding the full compensation package, not just base salary, including bonus structure, equity/stock, retirement matching, health benefits, remote/flexibility policy, learning budget, and PTO.

By following this approach, you can ensure a well-informed and str